# BAMoE — Bias-Aware Mixture of Experts for Time Series Forecasting

Run all five experiments end-to-end:
1. **Preliminary** — single-bias Transformer comparison
2. **Main results** — BAMoE vs. baselines
3. **Ablation** — expert diversity, routing mechanism, K-sweep
4. **Interpretability** — routing dynamics and attention patterns
5. **Efficiency** — parameter count vs. performance

> **Before running:** make sure the repo is public (or you have a token), and that `Runtime → Change runtime type` is set to **GPU**.
>
> Results are downloaded automatically after each experiment completes.

## 0 · Configuration
Edit the values in this cell before running anything else.

In [ ]:
# ── Repository ────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/hingma/BAMoE.git"
BRANCH      = "main"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Set USE_DRIVE=True to persist checkpoints/results across Colab sessions.
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/BAMoE"   # ignored when USE_DRIVE=False

# ── Config file ───────────────────────────────────────────────────────────────
# Which YAML to use as hyperparameter defaults.
# Choices: "configs/main.yaml" | "configs/preliminary.yaml" | "configs/ablation.yaml"
# The YAML is loaded in Section 3 after cloning the repo.
CONFIG_FILE = "configs/main.yaml"

# Per-notebook overrides — any key here wins over the YAML value.
CFG_OVERRIDES = dict(
    train_epochs = 100,   # uncomment for a quick smoke-test
)

# ── Weights & Biases ──────────────────────────────────────────────────────────
USE_WANDB     = True          # set True to enable run logging
WANDB_PROJECT = "BAMoE"
WANDB_ENTITY  = ""             # your wandb username / team, or "" for default

## 1 · Environment setup

In [ ]:
import subprocess, sys, os

# Reduce CUDA memory fragmentation across many sequential runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

IN_COLAB = "google.colab" in sys.modules

# GPU info
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print("GPU :", r.stdout.strip() if r.returncode == 0 else "none — running on CPU")
print("Python:", sys.version.split()[0])

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive mounted ->", DRIVE_DIR)


## 2 · Install dependencies

In [ ]:
!pip install torch numpy pandas scikit-learn matplotlib seaborn scipy tqdm wandb --quiet

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login()   # prompts for API key on first run; cached afterwards in ~/.netrc

## 3 · Clone repository

In [ ]:
WORK_DIR = DRIVE_DIR if (IN_COLAB and USE_DRIVE) else "/content/BAMoE"

if os.path.isdir(os.path.join(WORK_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {WORK_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {GITHUB_REPO} {WORK_DIR}

%cd {WORK_DIR}
print("Working directory:", os.getcwd())

## 4 · Download datasets

In [ ]:
!bash scripts/download_data.sh ./data

## 5 · Helpers

In [ ]:
import shutil, gc, random
import numpy as np
import torch
from run import build_args   # single source of truth for all arg defaults

if USE_WANDB:
    import wandb

# ── Per-dataset memory guards ─────────────────────────────────────────────────
# Traffic (862 cols) and Electricity (321 cols) need smaller batches.
DATASET_OVERRIDES = {
    "Traffic":     dict(batch_size=4,  num_workers=0),
    "Electricity": dict(batch_size=16, num_workers=0),
}

# ── Experiment runner ─────────────────────────────────────────────────────────

def make_args(overrides: dict):
    """Build an args Namespace from the active CONFIG_FILE + overrides."""
    merged = dict(
        root_path     = "./data",
        resume        = True,          # skip re-training if checkpoint exists
        use_amp       = True,
        wandb         = USE_WANDB,
        wandb_project = WANDB_PROJECT,
        wandb_entity  = WANDB_ENTITY or None,
    )
    merged.update(DATASET_OVERRIDES.get(overrides.get("data", ""), {}))
    merged.update(overrides)           # caller wins over everything above
    return build_args(CONFIG_FILE, **merged)


def set_seed(seed=2024):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def _free_gpu(exp):
    """Move model to CPU to immediately release GPU tensors, then GC."""
    try:
        exp.model.cpu()
    except Exception:
        pass
    del exp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        print(f"  GPU free after cleanup: {free_gb:.1f} GB")


def run_one(overrides: dict):
    """Train + test one configuration, then free GPU memory before returning."""
    from exp.exp_forecast import ExpForecast
    set_seed()
    args = make_args(overrides)
    print(f"\n{'='*60}\n{args.exp_name}  "
          f"[bs={args.batch_size}, workers={args.num_workers}, amp={args.use_amp}]")

    if USE_WANDB:
        wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None,
                   name=args.exp_name, config=vars(args), reinit=True)
    exp = None
    try:
        exp = ExpForecast(args)
        exp.train()
        result = exp.test()
    except torch.cuda.OutOfMemoryError:
        print("  OOM — cleaning up and skipping this run.")
        result = (float('nan'),) * 4
    finally:
        if USE_WANDB:
            wandb.finish()
        if exp is not None:
            _free_gpu(exp)
    return result


# ── Download helper ───────────────────────────────────────────────────────────

def download_results(label: str):
    """Zip results/ and trigger a browser download (Colab) or save locally."""
    zip_name = f"bamoe_{label}"
    shutil.make_archive(zip_name, "zip", "results")
    size_mb = os.path.getsize(f"{zip_name}.zip") / 1e6
    print(f"\n📦  {zip_name}.zip  ({size_mb:.1f} MB) — ", end="")
    if IN_COLAB:
        from google.colab import files
        files.download(f"{zip_name}.zip")
        print("download started.")
    else:
        print(f"saved to {os.path.abspath(zip_name + '.zip')}")


print("Helpers ready.")

---
## Experiment 1 — Preliminary: Temporal Inductive Bias Analysis

Trains four single-bias Transformers (global / causal / local / periodic) across
datasets and horizons to show that no single bias dominates — the key motivation
for BAMoE.

Adjust `EXP1_DATASETS` / `EXP1_PRED_LENS` to run a faster subset.

In [ ]:
EXP1_BIAS_TYPES = ["global", "causal", "local", "periodic"]
# EXP1_DATASETS   = ["ETTh1", "ETTh2", "Weather", "Traffic"]  # extend with ETTm1/m2, Electricity, Exchange
EXP1_DATASETS   = ["ETTh1", "ETTh2"]
# EXP1_PRED_LENS  = [96, 192, 336, 720]
EXP1_PRED_LENS  = [96]

for bias in EXP1_BIAS_TYPES:
    for data in EXP1_DATASETS:
        for pl in EXP1_PRED_LENS:
            run_one(dict(model="SingleBias", bias_type=bias, data=data, pred_len=pl))

download_results("exp1_preliminary")

---
## Experiment 2 — Main BAMoE Results

In [ ]:
# EXP2_DATASETS  = ["ETTh1", "ETTh2", "ETTm1", "ETTm2",
#                   "Weather", "Traffic", "Electricity", "Exchange"]
EXP2_DATASETS  = ["ETTh1", "ETTh2"]
# EXP2_PRED_LENS = [96, 192, 336, 720]
EXP2_PRED_LENS = [96]

for data in EXP2_DATASETS:
    for pl in EXP2_PRED_LENS:
        run_one(dict(model="BAMoE", data=data, pred_len=pl))

download_results("exp2_main")

---
## Experiment 3 — Ablation Studies
### 3a · Expert diversity

In [ ]:
# EXP3_DATASETS  = ["ETTh1", "Weather", "Traffic"]
EXP3_DATASETS  = ["ETTh1", "ETTh2"]
# EXP3_PRED_LENS = [96, 192, 336, 720]
EXP3_PRED_LENS = [96]

DIVERSITY_VARIANTS = {
    "homo_causal"   : "causal,causal,causal,causal",
    "homo_local"    : "local,local,local,local",
    "homo_periodic" : "periodic,periodic,periodic,periodic",
    "homo_global"   : "global,global,global,global",
    "K1"            : "causal",
    "hetero"        : "causal,local,periodic,global",
}

for tag, etypes in DIVERSITY_VARIANTS.items():
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", expert_types=etypes, data=data, pred_len=pl,
                         exp_name=f"ablation_diversity_{tag}_{data}_pl{pl}"))

download_results("exp3a_diversity")

### 3b · Routing mechanism

In [ ]:
ROUTING_VARIANTS = ["uniform", "random", "top1", "learned_sparse", "dense"]

for routing in ROUTING_VARIANTS:
    k = 1 if routing == "top1" else 2
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", routing=routing, top_k=k,
                         data=data, pred_len=pl,
                         exp_name=f"ablation_routing_{routing}_{data}_pl{pl}"))

download_results("exp3b_routing")

### 3c · Number of experts K

In [ ]:
K_VARIANTS = {
    "K2" : "causal,global",
    "K3" : "causal,local,periodic",
    "K4" : "causal,local,periodic,global",
    "K6" : "causal,local,periodic,global,causal,local",
    "K8" : "causal,local,periodic,global,causal,local,periodic,global",
}

for tag, etypes in K_VARIANTS.items():
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", expert_types=etypes,
                         data=data, pred_len=pl,
                         exp_name=f"ablation_Ksweep_{tag}_{data}_pl{pl}"))

download_results("exp3c_k_sweep")

---
## Experiment 4 — Interpretability Analysis
Requires completed Experiment 2 checkpoints.

In [ ]:
from exp.exp_interpretability import ExpInterpretability

EXP4_DATASETS  = ["ETTh1", "ETTh2", "Weather", "Traffic"]
EXP4_PRED_LENS = [96, 192, 336, 720]

for data in EXP4_DATASETS:
    for pl in EXP4_PRED_LENS:
        args = make_args(dict(model="BAMoE", data=data, pred_len=pl))
        ExpInterpretability(args).run()

download_results("exp4_interpretability")

---
## Results — Summary Table

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary.csv")
pivot = (
    df.groupby(["model", "data", "pred_len"])[["mse", "mae", "crps", "mase"]]
    .mean()
    .round(4)
)
pd.set_option("display.max_rows", 200)
pivot

In [ ]:
import matplotlib.pyplot as plt

avg = df.groupby("model")["mse"].mean().sort_values()
fig, ax = plt.subplots(figsize=(max(6, len(avg) * 0.8), 4))
avg.plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_ylabel("Average MSE")
ax.set_title("Average MSE per model (all datasets × horizons)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("results/avg_mse_bar.pdf", dpi=150)
plt.show()

download_results("final_all")